In [28]:
# Import các thư viện cần thiết
import sys
sys.path.append('../Common/')

import CommonSSI
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import backtrader as bt
import matplotlib.pyplot as plt


In [29]:
# Cấu hình tham số
symbol = 'SSI'  # Mã cổ phiếu SSI
from_date = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')  # 1 năm trước
to_date = datetime.now().strftime('%Y-%m-%d')  # Ngày hiện tại

print(f"Lấy dữ liệu {symbol} từ {from_date} đến {to_date}")


Lấy dữ liệu SSI từ 2024-11-19 đến 2025-11-19


In [30]:
# Lấy dữ liệu từ SSI
data = CommonSSI.CommonSSI.loaddataSSI(symbol, from_date, to_date)

# Chuyển đổi Datetime sang datetime object và set làm index
data['Datetime'] = pd.to_datetime(data['Datetime'], format='%d/%m/%Y')
data.set_index('Datetime', inplace=True)

# Sắp xếp theo thời gian
data.sort_index(inplace=True)

print(f"Đã lấy được {len(data)} ngày giao dịch")
print(data.head())
print(data.tail())


daily_ohlc(symbol='SSI', fromDate='19/11/2024', toDate='19/11/2025', pageIndex=1, pageSize=100, ascending=True)
{'data': [{'Symbol': 'SSI', 'Market': 'HOSE', 'TradingDate': '19/11/2024', 'Time': None, 'Open': '23746', 'High': '23892', 'Low': '23210', 'Close': '23259', 'Volume': '10705300', 'Value': '257976089999.9990'}, {'Symbol': 'SSI', 'Market': 'HOSE', 'TradingDate': '20/11/2024', 'Time': None, 'Open': '23307', 'High': '24087', 'Low': '23161', 'Close': '23697', 'Volume': '17961700', 'Value': '436186615000'}, {'Symbol': 'SSI', 'Market': 'HOSE', 'TradingDate': '21/11/2024', 'Time': None, 'Open': '23697', 'High': '24039', 'Low': '23454', 'Close': '23990', 'Volume': '14195600', 'Value': '344864820000.0060'}, {'Symbol': 'SSI', 'Market': 'HOSE', 'TradingDate': '22/11/2024', 'Time': None, 'Open': '23892', 'High': '23990', 'Low': '23649', 'Close': '23746', 'Volume': '12804200', 'Value': '312167864999.9960'}, {'Symbol': 'SSI', 'Market': 'HOSE', 'TradingDate': '25/11/2024', 'Time': None, 'Ope

In [31]:
# Tính toán MA10 và MA20
data['MA10'] = data['Close'].rolling(window=10).mean()
data['MA20'] = data['Close'].rolling(window=20).mean()

# Chuyen kieu du lieu cua cac cot Open, High, Low, Close, Volume thanh float
data['Open'] = data['Open'].astype(float)
data['High'] = data['High'].astype(float)
data['Low'] = data['Low'].astype(float)
data['Close'] = data['Close'].astype(float)
data['Volume'] = data['Volume'].astype(float)

# Tạo tín hiệu Buy/Sell
# MA10 > MA20 thì Buy, ngược lại thì Sell
data['Buy_Signal'] = (data['MA10'] > data['MA20']).astype(int)
data['Sell_Signal'] = (data['MA10'] <= data['MA20']).astype(int)

# Hiển thị một số dòng để kiểm tra
print("Dữ liệu với MA và tín hiệu:")
print(data[['Close', 'MA10', 'MA20', 'Buy_Signal', 'Sell_Signal']].tail(20))


Dữ liệu với MA và tín hiệu:
              Close     MA10      MA20  Buy_Signal  Sell_Signal
Datetime                                                       
2025-03-19  25892.0  26145.2  25730.75           1            0
2025-03-20  25892.0  26111.1  25794.15           1            0
2025-03-21  25940.0  26081.8  25855.10           1            0
2025-03-24  26477.0  26130.6  25918.50           1            0
2025-03-25  26233.0  26150.1  25969.70           1            0
2025-03-26  25843.0  26130.6  26001.40           1            0
2025-03-27  25648.0  26067.2  26011.15           1            0
2025-03-28  25697.0  26008.7  26013.60           0            1
2025-03-31  25355.0  25911.2  25984.35           0            1
2025-04-01  25404.0  25838.1  25969.70           0            1
2025-04-02  25794.0  25828.3  25986.75           0            1
2025-04-03  23990.0  25638.1  25874.60           0            1
2025-04-04  23161.0  25360.2  25721.00           0            1
2025-04-08  

In [ ]:
# Định nghĩa chiến lược Backtrader
class MAStrategy(bt.Strategy):
    params = (
        ('ma_fast', 10),
        ('ma_slow', 20),
    )
    
    def __init__(self):
        # Tính toán MA10 và MA20
        self.ma_fast = bt.indicators.SMA(self.data.close, period=self.params.ma_fast)
        self.ma_slow = bt.indicators.SMA(self.data.close, period=self.params.ma_slow)
        
        # Tín hiệu giao cắt
        self.crossover = bt.indicators.CrossOver(self.ma_fast, self.ma_slow)
        
        # Lưu danh sách các lệnh để phân tích sau
        self.trade_list = []
        
        # Theo dõi các tín hiệu bán bị bỏ qua (khi chưa có cổ phiếu)
        self.skipped_sells = []
        
    def notify_trade(self, trade):
        # Lưu thông tin lệnh khi lệnh đóng
        if trade.isclosed:
            self.trade_list.append({
                'entry_date': trade.dtopen,
                'exit_date': trade.dtclose,
                'entry_price': trade.price,
                'exit_price': trade.pnl / trade.size + trade.price if trade.size != 0 else 0,
                'size': trade.size,
                'pnl': trade.pnl,
                'pnlcomm': trade.pnlcomm,
                'result': 'WIN' if trade.pnlcomm > 0 else 'LOSS' if trade.pnlcomm < 0 else 'BREAK'
            })
        
    def next(self):
        # Nếu MA10 cắt lên trên MA20 (crossover > 0) thì mua
        if self.crossover > 0:
            if not self.position:
                self.buy()
                print(f'{self.data.datetime.date(0)}: MUA @ {self.data.close[0]:.2f}')
        
        # Nếu MA10 cắt xuống dưới MA20 (crossover < 0) thì bán
        elif self.crossover < 0:
            if self.position:
                # Có cổ phiếu → Bán được
                self.sell()
                print(f'{self.data.datetime.date(0)}: BÁN @ {self.data.close[0]:.2f}')
            else:
                # KHÔNG có cổ phiếu → KHÔNG thể bán → Bỏ qua tín hiệu
                self.skipped_sells.append({
                    'date': self.data.datetime.date(0),
                    'price': self.data.close[0],
                    'reason': 'Không có cổ phiếu để bán'
                })
                # Không in ra để tránh spam, sẽ thống kê sau


In [33]:
# Chuẩn bị dữ liệu cho Backtrader
# Backtrader cần dữ liệu theo format: datetime, open, high, low, close, volume, openinterest
# Đảm bảo các cột có đúng tên và kiểu dữ liệu
data_bt = data[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
data_bt.columns = ['open', 'high', 'low', 'close', 'volume']

# Đảm bảo tất cả các cột đều là số (float)
for col in ['open', 'high', 'low', 'close', 'volume']:
    data_bt[col] = pd.to_numeric(data_bt[col], errors='coerce')

# Thêm cột openinterest (cần thiết cho Backtrader, có thể để = 0)
data_bt['openinterest'] = 0

# Đảm bảo dữ liệu không có NaN
data_bt = data_bt.dropna()

# Đảm bảo index là datetime và được sắp xếp
data_bt.index = pd.to_datetime(data_bt.index)
data_bt = data_bt.sort_index()

print(f"Dữ liệu đã sẵn sàng cho Backtrader: {len(data_bt)} ngày")
print(f"Ngày bắt đầu: {data_bt.index[0]}")
print(f"Ngày kết thúc: {data_bt.index[-1]}")
print(data_bt.head())
print("\nKiểu dữ liệu:")
print(data_bt.dtypes)


Dữ liệu đã sẵn sàng cho Backtrader: 100 ngày
Ngày bắt đầu: 2024-11-19 00:00:00
Ngày kết thúc: 2025-04-16 00:00:00
               open     high      low    close      volume  openinterest
Datetime                                                                
2024-11-19  23746.0  23892.0  23210.0  23259.0  10705300.0             0
2024-11-20  23307.0  24087.0  23161.0  23697.0  17961700.0             0
2024-11-21  23697.0  24039.0  23454.0  23990.0  14195600.0             0
2024-11-22  23892.0  23990.0  23649.0  23746.0  12804200.0             0
2024-11-25  23795.0  23892.0  23600.0  23892.0  10156700.0             0

Kiểu dữ liệu:
open            float64
high            float64
low             float64
close           float64
volume          float64
openinterest      int64
dtype: object


In [34]:
# Tạo Cerebro engine (Backtrader)
cerebro = bt.Cerebro()

# Thêm dữ liệu vào Cerebro
# PandasData sẽ tự động sử dụng index làm datetime nếu index là datetime
# và các cột có tên đúng: open, high, low, close, volume, openinterest
datafeed = bt.feeds.PandasData(dataname=data_bt)
cerebro.adddata(datafeed)

# Thêm chiến lược
cerebro.addstrategy(MAStrategy)

# Thêm các Analyzers để tính toán thống kê giao dịch
cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')  # Phân tích từng lệnh
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')  # Sharpe Ratio
cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')  # Drawdown
cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')  # Returns

# Thiết lập vốn ban đầu
initial_cash = 100000000  # 100 triệu VNĐ
cerebro.broker.setcash(initial_cash)

# Thiết lập phí giao dịch (0.15% cho mua và bán)
cerebro.broker.setcommission(commission=0.0015)

# Thiết lập kích thước lệnh (mua/bán theo số lượng cổ phiếu)
cerebro.addsizer(bt.sizers.FixedSize, stake=100)  # Mỗi lần mua/bán 100 cổ phiếu

print(f'Vốn ban đầu: {cerebro.broker.getvalue():,.0f} VNĐ')


Vốn ban đầu: 100,000,000 VNĐ


In [35]:
# Chạy backtest
print('=' * 50)
print('BẮT ĐẦU BACKTEST')
print('=' * 50)

# Chạy backtest và lấy kết quả
strats = cerebro.run()

# Lấy strategy từ kết quả
strategy = strats[0]

# Lấy giá trị cuối cùng
final_value = cerebro.broker.getvalue()
profit = final_value - initial_cash
profit_pct = (profit / initial_cash) * 100

print('=' * 50)
print('KẾT QUẢ BACKTEST')
print('=' * 50)
print(f'Vốn ban đầu: {initial_cash:,.0f} VNĐ')
print(f'Vốn cuối cùng: {final_value:,.0f} VNĐ')
print(f'Lợi nhuận: {profit:,.0f} VNĐ')
print(f'Lợi nhuận (%): {profit_pct:.2f}%')


BẮT ĐẦU BACKTEST
2025-02-05: MUA @ 24429.00
2025-03-28: BÁN @ 25697.00
KẾT QUẢ BACKTEST
Vốn ban đầu: 100,000,000 VNĐ
Vốn cuối cùng: 100,085,089 VNĐ
Lợi nhuận: 85,089 VNĐ
Lợi nhuận (%): 0.09%


# 📚 GIẢI THÍCH: KHI NÀO LỆNH THẮNG, KHI NÀO LỆNH THUA?

## 🔵 GIAO DỊCH CHỨNG KHOÁN (MUA/BÁN CỔ PHIẾU)

### 1. CÁCH THỨC GIAO DỊCH:
- **MUA (BUY)**: Mua cổ phiếu ở một mức giá → Nắm giữ → Bán ở một mức giá khác
- **BÁN (SELL)**: Bán cổ phiếu đã mua để chốt lời/lỗ

### 2. CÔNG THỨC TÍNH LỢI NHUẬN/LỖ:

```
Lợi nhuận/Lỗ (P&L) = (Giá Bán - Giá Mua) × Số lượng cổ phiếu - Phí giao dịch
```

**Chi tiết:**
- **Giá Mua**: Giá mua cổ phiếu (ví dụ: 24,429 VNĐ/cổ phiếu)
- **Giá Bán**: Giá bán cổ phiếu (ví dụ: 25,697 VNĐ/cổ phiếu)
- **Số lượng**: Số cổ phiếu mua/bán (ví dụ: 100 cổ phiếu)
- **Phí giao dịch**: 
  - Phí mua: 0.15% × (Giá Mua × Số lượng)
  - Phí bán: 0.15% × (Giá Bán × Số lượng)

### 3. KHI NÀO LỆNH THẮNG? ✅

**Lệnh THẮNG khi:**
```
Giá Bán > Giá Mua (sau khi trừ phí giao dịch)
```

**Ví dụ:**
- Mua: 100 cổ phiếu @ 24,429 VNĐ = 2,442,900 VNĐ
- Phí mua: 0.15% × 2,442,900 = 3,664 VNĐ
- **Tổng chi phí mua: 2,446,564 VNĐ**

- Bán: 100 cổ phiếu @ 25,697 VNĐ = 2,569,700 VNĐ
- Phí bán: 0.15% × 2,569,700 = 3,855 VNĐ
- **Tổng thu được: 2,565,845 VNĐ**

- **Lợi nhuận = 2,565,845 - 2,446,564 = 119,281 VNĐ** ✅ **THẮNG**

### 4. KHI NÀO LỆNH THUA? ❌

**Lệnh THUA khi:**
```
Giá Bán < Giá Mua (sau khi trừ phí giao dịch)
```

**Ví dụ:**
- Mua: 100 cổ phiếu @ 25,697 VNĐ = 2,569,700 VNĐ
- Phí mua: 0.15% × 2,569,700 = 3,855 VNĐ
- **Tổng chi phí mua: 2,573,555 VNĐ**

- Bán: 100 cổ phiếu @ 24,429 VNĐ = 2,442,900 VNĐ
- Phí bán: 0.15% × 2,442,900 = 3,664 VNĐ
- **Tổng thu được: 2,439,236 VNĐ**

- **Lỗ = 2,439,236 - 2,573,555 = -134,319 VNĐ** ❌ **THUA**

### 5. TRONG BACKTRADER:

Backtrader tự động tính:
- **`trade.pnl`**: Lợi nhuận/lỗ trước phí (Gross P&L)
- **`trade.pnlcomm`**: Lợi nhuận/lỗ sau phí (Net P&L) - **Đây là giá trị quyết định thắng/thua**

**Quy tắc:**
- **`pnlcomm > 0`** → ✅ **LỆNH THẮNG**
- **`pnlcomm < 0`** → ❌ **LỆNH THUA**
- **`pnlcomm = 0`** → ➖ **HÒA VỐN**

### 6. LƯU Ý QUAN TRỌNG:

⚠️ **Phí giao dịch rất quan trọng!**
- Ngay cả khi giá tăng, nếu tăng ít hơn tổng phí giao dịch → Vẫn LỖ
- Ví dụ: Mua 24,429, bán 24,500 (tăng 71 VNĐ) nhưng phí = 73 VNĐ → Vẫn LỖ

⚠️ **Trong chiến lược MA10/MA20:**
- Mua khi MA10 cắt lên trên MA20 (tín hiệu tăng giá)
- Bán khi MA10 cắt xuống dưới MA20 (tín hiệu giảm giá)
- Mục tiêu: Bắt được xu hướng tăng và bán trước khi giá giảm mạnh


In [36]:
# Ví dụ minh họa cách tính lệnh thắng/thua
print('=' * 70)
print('VÍ DỤ MINH HỌA: CÁCH TÍNH LỆNH THẮNG/THUA')
print('=' * 70)

# Lấy một lệnh mẫu từ kết quả backtest (nếu có)
if hasattr(strategy, 'trade_list') and len(strategy.trade_list) > 0:
    sample_trade = strategy.trade_list[0]
    
    print(f"\n📊 VÍ DỤ LỆNH GIAO DỊCH:")
    print(f"   Mã cổ phiếu: {symbol}")
    print(f"   Số lượng: {sample_trade['size']:.0f} cổ phiếu")
    print(f"   Ngày mua: {sample_trade['entry_date']}")
    print(f"   Giá mua: {sample_trade['entry_price']:,.0f} VNĐ/cổ phiếu")
    print(f"   Ngày bán: {sample_trade['exit_date']}")
    print(f"   Giá bán: {sample_trade['exit_price']:,.0f} VNĐ/cổ phiếu")
    
    # Tính toán chi tiết
    entry_price = sample_trade['entry_price']
    exit_price = sample_trade['exit_price']
    size = sample_trade['size']
    commission_rate = 0.0015  # 0.15%
    
    # Tính phí
    buy_cost = entry_price * size
    buy_commission = buy_cost * commission_rate
    total_buy_cost = buy_cost + buy_commission
    
    sell_revenue = exit_price * size
    sell_commission = sell_revenue * commission_rate
    total_sell_revenue = sell_revenue - sell_commission
    
    # Lợi nhuận
    gross_pnl = (exit_price - entry_price) * size
    net_pnl = total_sell_revenue - total_buy_cost
    
    print(f"\n💰 TÍNH TOÁN CHI TIẾT:")
    print(f"\n   BƯỚC 1: MUA CỔ PHIẾU")
    print(f"   - Giá mua: {entry_price:,.0f} VNĐ × {size:.0f} = {buy_cost:,.0f} VNĐ")
    print(f"   - Phí mua (0.15%): {buy_commission:,.0f} VNĐ")
    print(f"   → Tổng chi phí mua: {total_buy_cost:,.0f} VNĐ")
    
    print(f"\n   BƯỚC 2: BÁN CỔ PHIẾU")
    print(f"   - Giá bán: {exit_price:,.0f} VNĐ × {size:.0f} = {sell_revenue:,.0f} VNĐ")
    print(f"   - Phí bán (0.15%): {sell_commission:,.0f} VNĐ")
    print(f"   → Tổng thu được: {total_sell_revenue:,.0f} VNĐ")
    
    print(f"\n   BƯỚC 3: TÍNH LỢI NHUẬN/LỖ")
    print(f"   - Lợi nhuận trước phí (Gross P&L): {gross_pnl:,.0f} VNĐ")
    print(f"   - Tổng phí giao dịch: {buy_commission + sell_commission:,.0f} VNĐ")
    print(f"   - Lợi nhuận sau phí (Net P&L): {net_pnl:,.0f} VNĐ")
    
    print(f"\n   🎯 KẾT QUẢ:")
    if net_pnl > 0:
        print(f"   ✅ LỆNH THẮNG! (Lợi nhuận: {net_pnl:,.0f} VNĐ)")
        print(f"   → Vì: Giá bán ({exit_price:,.0f}) > Giá mua ({entry_price:,.0f})")
        print(f"   → Sau khi trừ phí, vẫn còn lợi nhuận")
    elif net_pnl < 0:
        print(f"   ❌ LỆNH THUA! (Lỗ: {abs(net_pnl):,.0f} VNĐ)")
        print(f"   → Vì: Giá bán ({exit_price:,.0f}) < Giá mua ({entry_price:,.0f})")
        print(f"   → Hoặc: Lợi nhuận không đủ bù phí giao dịch")
    else:
        print(f"   ➖ HÒA VỐN (Không lời không lỗ)")
    
    print(f"\n   📝 LƯU Ý:")
    print(f"   - Backtrader sử dụng giá trị 'pnlcomm' ({sample_trade['pnlcomm']:,.0f} VNĐ) để xác định thắng/thua")
    print(f"   - Giá trị này đã bao gồm tất cả phí giao dịch")
    
else:
    print("\n⚠️ Chưa có lệnh giao dịch nào để minh họa.")
    print("   Hãy chạy backtest trước để xem ví dụ.")
    
    # Ví dụ giả định
    print("\n📊 VÍ DỤ GIẢ ĐỊNH:")
    print("\n   Tình huống: Mua 100 cổ phiếu SSI")
    print("   - Giá mua: 24,429 VNĐ/cổ phiếu")
    print("   - Giá bán: 25,697 VNĐ/cổ phiếu")
    print("   - Phí giao dịch: 0.15% mỗi lần")
    
    entry_price = 24429
    exit_price = 25697
    size = 100
    commission_rate = 0.0015
    
    buy_cost = entry_price * size
    buy_commission = buy_cost * commission_rate
    total_buy_cost = buy_cost + buy_commission
    
    sell_revenue = exit_price * size
    sell_commission = sell_revenue * commission_rate
    total_sell_revenue = sell_revenue - sell_commission
    
    net_pnl = total_sell_revenue - total_buy_cost
    
    print(f"\n   Tính toán:")
    print(f"   - Chi phí mua: {buy_cost:,.0f} + {buy_commission:,.0f} (phí) = {total_buy_cost:,.0f} VNĐ")
    print(f"   - Thu được: {sell_revenue:,.0f} - {sell_commission:,.0f} (phí) = {total_sell_revenue:,.0f} VNĐ")
    print(f"   - Lợi nhuận: {net_pnl:,.0f} VNĐ")
    
    if net_pnl > 0:
        print(f"\n   ✅ KẾT QUẢ: LỆNH THẮNG!")
    else:
        print(f"\n   ❌ KẾT QUẢ: LỆNH THUA!")


VÍ DỤ MINH HỌA: CÁCH TÍNH LỆNH THẮNG/THUA

📊 VÍ DỤ LỆNH GIAO DỊCH:
   Mã cổ phiếu: SSI
   Số lượng: 0 cổ phiếu
   Ngày mua: 739288.0
   Giá mua: 24,575 VNĐ/cổ phiếu
   Ngày bán: 739341.0
   Giá bán: 0 VNĐ/cổ phiếu

💰 TÍNH TOÁN CHI TIẾT:

   BƯỚC 1: MUA CỔ PHIẾU
   - Giá mua: 24,575 VNĐ × 0 = 0 VNĐ
   - Phí mua (0.15%): 0 VNĐ
   → Tổng chi phí mua: 0 VNĐ

   BƯỚC 2: BÁN CỔ PHIẾU
   - Giá bán: 0 VNĐ × 0 = 0 VNĐ
   - Phí bán (0.15%): 0 VNĐ
   → Tổng thu được: 0 VNĐ

   BƯỚC 3: TÍNH LỢI NHUẬN/LỖ
   - Lợi nhuận trước phí (Gross P&L): -0 VNĐ
   - Tổng phí giao dịch: 0 VNĐ
   - Lợi nhuận sau phí (Net P&L): 0 VNĐ

   🎯 KẾT QUẢ:
   ➖ HÒA VỐN (Không lời không lỗ)

   📝 LƯU Ý:
   - Backtrader sử dụng giá trị 'pnlcomm' (85,089 VNĐ) để xác định thắng/thua
   - Giá trị này đã bao gồm tất cả phí giao dịch


# ⚠️ QUAN TRỌNG: NẾU TÍN HIỆU BÁN XUẤT HIỆN ĐẦU TIÊN?

## 🔴 TRẢ LỜI: KHÔNG THỰC HIỆN VÀ KHÔNG TÍNH PNL

### 1. TẠI SAO KHÔNG THỂ BÁN?

Trong giao dịch chứng khoán thông thường (không bán khống):
- **Bạn PHẢI có cổ phiếu trước khi bán**
- Không thể bán cái bạn không có!

### 2. LOGIC TRONG CODE:

```python
elif self.crossover < 0:  # Tín hiệu bán
    if self.position:     # NẾU có cổ phiếu
        self.sell()        # → Bán được
    else:                  # NẾU KHÔNG có cổ phiếu
        # → KHÔNG bán, bỏ qua tín hiệu
```

### 3. ĐIỀU GÌ XẢY RA?

**Kịch bản: Tín hiệu bán xuất hiện đầu tiên**
1. ✅ Tín hiệu bán được phát hiện (MA10 < MA20)
2. ❌ Kiểm tra: `if self.position:` → **FALSE** (chưa có cổ phiếu)
3. ⏭️ **BỎ QUA** tín hiệu bán
4. ❌ **KHÔNG có giao dịch nào xảy ra**
5. ❌ **KHÔNG tính PNL** (vì không có giao dịch)

### 4. VÍ DỤ:

**Ngày 1:** MA10 < MA20 → Tín hiệu BÁN
- ❌ Không có cổ phiếu → Bỏ qua
- ❌ Không có giao dịch
- ❌ Không tính PNL

**Ngày 5:** MA10 > MA20 → Tín hiệu MUA
- ✅ Không có cổ phiếu → Mua được
- ✅ Mua 100 cổ phiếu @ 24,429 VNĐ

**Ngày 10:** MA10 < MA20 → Tín hiệu BÁN
- ✅ Có cổ phiếu → Bán được
- ✅ Bán 100 cổ phiếu @ 25,697 VNĐ
- ✅ Tính PNL: (25,697 - 24,429) × 100 - phí = +119,281 VNĐ

### 5. LƯU Ý:

⚠️ **Trong chiến lược MA10/MA20:**
- Luôn bắt đầu bằng tín hiệu MUA (khi MA10 cắt lên)
- Sau đó mới có tín hiệu BÁN (khi MA10 cắt xuống)
- Nếu tín hiệu BÁN xuất hiện đầu tiên → Bỏ qua, chờ tín hiệu MUA

⚠️ **Backtrader tự động xử lý:**
- Chỉ thực hiện lệnh khi có điều kiện hợp lệ
- Không tính PNL cho các lệnh không được thực hiện


In [ ]:
# Kiểm tra các tín hiệu bán bị bỏ qua (nếu có)
print('=' * 70)
print('KIỂM TRA: CÁC TÍN HIỆU BÁN BỊ BỎ QUA')
print('=' * 70)

if hasattr(strategy, 'skipped_sells') and len(strategy.skipped_sells) > 0:
    print(f"\n⚠️ Có {len(strategy.skipped_sells)} tín hiệu BÁN bị bỏ qua vì chưa có cổ phiếu:")
    print("\n" + "-" * 70)
    print(f"{'STT':<5} {'Ngày':<15} {'Giá':<15} {'Lý do':<30}")
    print("-" * 70)
    
    for i, skipped in enumerate(strategy.skipped_sells, 1):
        print(f"{i:<5} {str(skipped['date']):<15} {skipped['price']:>14,.0f} {skipped['reason']:<30}")
    
    print("-" * 70)
    print(f"\n📝 Giải thích:")
    print(f"   - Các tín hiệu bán này KHÔNG được thực hiện")
    print(f"   - Vì chưa có cổ phiếu trong tài khoản")
    print(f"   - KHÔNG có giao dịch → KHÔNG tính PNL")
    print(f"   - Đây là hành vi ĐÚNG trong giao dịch chứng khoán")
    
else:
    print("\n✅ Không có tín hiệu bán nào bị bỏ qua.")
    print("   Tất cả các tín hiệu bán đều được thực hiện sau khi đã mua cổ phiếu.")
    
# So sánh số lệnh thực hiện
if hasattr(strategy, 'trade_list'):
    total_trades = len(strategy.trade_list)
    print(f"\n📊 Thống kê:")
    print(f"   - Số lệnh đã thực hiện (MUA-BÁN hoàn chỉnh): {total_trades}")
    if hasattr(strategy, 'skipped_sells'):
        print(f"   - Số tín hiệu bán bị bỏ qua: {len(strategy.skipped_sells)}")
    
    # Kiểm tra xem có tín hiệu bán nào xuất hiện trước lệnh mua đầu tiên không
    if total_trades > 0 and hasattr(strategy, 'skipped_sells') and len(strategy.skipped_sells) > 0:
        first_trade_date = strategy.trade_list[0]['entry_date']
        first_skipped_sell = strategy.skipped_sells[0]['date']
        
        if first_skipped_sell < first_trade_date:
            print(f"\n⚠️ LƯU Ý:")
            print(f"   - Tín hiệu bán đầu tiên ({first_skipped_sell}) xuất hiện TRƯỚC lệnh mua đầu tiên ({first_trade_date})")
            print(f"   - Điều này là BÌNH THƯỜNG và ĐÚNG")
            print(f"   - Chiến lược sẽ bỏ qua và chờ tín hiệu mua")


In [37]:
# Vẽ biểu đồ kết quả
# Tắt cảnh báo matplotlib
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Vẽ biểu đồ với style đơn giản hơn để tránh cảnh báo
try:
    cerebro.plot(style='candlestick', volume=True, figsize=(16, 10))
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Lỗi khi vẽ biểu đồ: {e}")
    # Thử vẽ với style khác
    cerebro.plot(style='line', volume=False, figsize=(16, 10))
    plt.tight_layout()
    plt.show()


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
# Tính toán và hiển thị các chỉ số hiệu suất chi tiết hơn
# So sánh với Buy & Hold
buy_hold_return = ((data['Close'].iloc[-1] - data['Close'].iloc[0]) / data['Close'].iloc[0]) * 100

print('=' * 50)
print('SO SÁNH VỚI BUY & HOLD')
print('=' * 50)
print(f'Chiến lược MA10/MA20: {profit_pct:.2f}%')
print(f'Buy & Hold: {buy_hold_return:.2f}%')
print(f'Chênh lệch: {profit_pct - buy_hold_return:.2f}%')


SO SÁNH VỚI BUY & HOLD
Chiến lược MA10/MA20: 0.09%
Buy & Hold: -3.56%
Chênh lệch: 3.65%


In [39]:
# Phân tích chi tiết các lệnh giao dịch
print('=' * 70)
print('PHÂN TÍCH CHI TIẾT CÁC LỆNH GIAO DỊCH')
print('=' * 70)

# Lấy TradeAnalyzer
trades = strategy.analyzers.trades.get_analysis()

if trades['total']['total'] > 0:
    total_trades = trades['total']['total']
    won = trades['won']['total']
    lost = trades['lost']['total']
    
    # Tính tỷ lệ thắng
    win_rate = (won / total_trades * 100) if total_trades > 0 else 0
    
    # Lợi nhuận trung bình
    avg_win = trades['won']['pnl']['average'] if won > 0 else 0
    avg_loss = trades['lost']['pnl']['average'] if lost > 0 else 0
    
    # Lợi nhuận/lỗ lớn nhất
    max_win = trades['won']['pnl']['max'] if won > 0 else 0
    max_loss = trades['lost']['pnl']['min'] if lost > 0 else 0
    
    # Tổng lợi nhuận từ lệnh thắng và lỗ từ lệnh thua
    total_won_pnl = trades['won']['pnl']['total'] if won > 0 else 0
    total_lost_pnl = trades['lost']['pnl']['total'] if lost > 0 else 0
    
    # Profit Factor = Tổng lợi nhuận / Tổng lỗ (giá trị tuyệt đối)
    profit_factor = abs(total_won_pnl / total_lost_pnl) if total_lost_pnl != 0 else float('inf')
    
    print(f"\n📊 TỔNG QUAN:")
    print(f"   Tổng số lệnh: {total_trades}")
    print(f"   Số lệnh THẮNG: {won} ({win_rate:.2f}%)")
    print(f"   Số lệnh THUA: {lost} ({100-win_rate:.2f}%)")
    
    print(f"\n💰 LỢI NHUẬN/LỖ:")
    print(f"   Tổng lợi nhuận từ lệnh thắng: {total_won_pnl:,.0f} VNĐ")
    print(f"   Tổng lỗ từ lệnh thua: {total_lost_pnl:,.0f} VNĐ")
    print(f"   Lợi nhuận trung bình mỗi lệnh thắng: {avg_win:,.0f} VNĐ")
    print(f"   Lỗ trung bình mỗi lệnh thua: {avg_loss:,.0f} VNĐ")
    print(f"   Lợi nhuận lớn nhất: {max_win:,.0f} VNĐ")
    print(f"   Lỗ lớn nhất: {max_loss:,.0f} VNĐ")
    
    print(f"\n📈 CHỈ SỐ HIỆU SUẤT:")
    print(f"   Win Rate (Tỷ lệ thắng): {win_rate:.2f}%")
    print(f"   Profit Factor: {profit_factor:.2f}")
    
    # Risk/Reward Ratio
    if avg_loss != 0:
        risk_reward = abs(avg_win / avg_loss) if avg_loss < 0 else 0
        print(f"   Risk/Reward Ratio: {risk_reward:.2f}")
    
    # Expectancy (Kỳ vọng lợi nhuận mỗi lệnh)
    expectancy = (win_rate/100 * avg_win) + ((100-win_rate)/100 * avg_loss)
    print(f"   Expectancy (Kỳ vọng/lệnh): {expectancy:,.0f} VNĐ")
    
    # Thống kê số lượng cổ phiếu
    if 'bought' in trades['total']:
        print(f"\n📦 SỐ LƯỢNG:")
        print(f"   Tổng số cổ phiếu đã mua: {trades['total']['bought']:.0f}")
        print(f"   Tổng số cổ phiếu đã bán: {trades['total']['sold']:.0f}")
    
else:
    print("⚠️ Không có lệnh giao dịch nào được thực hiện!")


PHÂN TÍCH CHI TIẾT CÁC LỆNH GIAO DỊCH

📊 TỔNG QUAN:
   Tổng số lệnh: 1
   Số lệnh THẮNG: 1 (100.00%)
   Số lệnh THUA: 0 (0.00%)

💰 LỢI NHUẬN/LỖ:
   Tổng lợi nhuận từ lệnh thắng: 85,089 VNĐ
   Tổng lỗ từ lệnh thua: 0 VNĐ
   Lợi nhuận trung bình mỗi lệnh thắng: 85,089 VNĐ
   Lỗ trung bình mỗi lệnh thua: 0 VNĐ
   Lợi nhuận lớn nhất: 85,089 VNĐ
   Lỗ lớn nhất: 0 VNĐ

📈 CHỈ SỐ HIỆU SUẤT:
   Win Rate (Tỷ lệ thắng): 100.00%
   Profit Factor: inf
   Expectancy (Kỳ vọng/lệnh): 85,089 VNĐ


In [40]:
# Phân tích Drawdown và các chỉ số rủi ro
print('=' * 70)
print('PHÂN TÍCH RỦI RO VÀ DRAWDOWN')
print('=' * 70)

drawdown = strategy.analyzers.drawdown.get_analysis()
returns = strategy.analyzers.returns.get_analysis()

if drawdown:
    print(f"\n📉 DRAWDOWN:")
    print(f"   Max Drawdown: {drawdown['max']['drawdown']:.2f}%")
    print(f"   Max Drawdown Period: {drawdown['max']['len']} ngày")
    if 'moneydown' in drawdown['max']:
        print(f"   Max Money Drawdown: {drawdown['max']['moneydown']:,.0f} VNĐ")

if returns:
    print(f"\n📊 RETURNS:")
    if 'rnorm100' in returns:
        print(f"   Return (normalized): {returns['rnorm100']:.2f}%")
    if 'rtot' in returns:
        print(f"   Total Return: {returns['rtot']:.2f}%")

# Sharpe Ratio
sharpe = strategy.analyzers.sharpe.get_analysis()
if sharpe and 'sharperatio' in sharpe:
    sharpe_ratio = sharpe['sharperatio']
    if sharpe_ratio is not None and not np.isnan(sharpe_ratio):
        print(f"\n📈 SHARPE RATIO:")
        print(f"   Sharpe Ratio: {sharpe_ratio:.2f}")
        if sharpe_ratio > 1:
            print("   ✅ Tốt (Sharpe > 1)")
        elif sharpe_ratio > 0:
            print("   ⚠️ Trung bình (0 < Sharpe < 1)")
        else:
            print("   ❌ Kém (Sharpe < 0)")


PHÂN TÍCH RỦI RO VÀ DRAWDOWN

📉 DRAWDOWN:
   Max Drawdown: 0.10%
   Max Drawdown Period: 16 ngày
   Max Money Drawdown: 101,425 VNĐ

📊 RETURNS:
   Return (normalized): 0.21%
   Total Return: 0.00%

📈 SHARPE RATIO:
   Sharpe Ratio: -22.50
   ❌ Kém (Sharpe < 0)


In [41]:
# Hiển thị chi tiết từng lệnh giao dịch
print('=' * 70)
print('CHI TIẾT TỪNG LỆNH GIAO DỊCH')
print('=' * 70)

# Sử dụng trade_list từ strategy hoặc từ analyzer
if hasattr(strategy, 'trade_list') and len(strategy.trade_list) > 0:
    trades = strategy.trade_list
    print(f"\nTổng số lệnh đã đóng: {len(trades)}")
    print("\n" + "-" * 90)
    print(f"{'STT':<5} {'Ngày Mua':<12} {'Giá Mua':<15} {'Ngày Bán':<12} {'Giá Bán':<15} {'Lợi nhuận':<15} {'Kết quả':<10}")
    print("-" * 90)
    
    for i, trade in enumerate(trades, 1):
        entry_date = trade['entry_date'].date() if hasattr(trade['entry_date'], 'date') else str(trade['entry_date'])
        entry_price = trade['entry_price']
        exit_date = trade['exit_date'].date() if hasattr(trade['exit_date'], 'date') else str(trade['exit_date'])
        exit_price = trade['exit_price']
        pnlcomm = trade['pnlcomm']
        result_text = trade['result']
        
        result = "✅ THẮNG" if result_text == 'WIN' else "❌ THUA" if result_text == 'LOSS' else "➖ HÒA"
        
        print(f"{i:<5} {str(entry_date):<12} {entry_price:>14,.0f} {str(exit_date):<12} {exit_price:>14,.0f} {pnlcomm:>14,.0f} {result:<10}")
    
    print("-" * 90)
    
    # Tóm tắt
    wins = sum(1 for t in trades if t['result'] == 'WIN')
    losses = sum(1 for t in trades if t['result'] == 'LOSS')
    print(f"\nTóm tắt: {wins} lệnh thắng, {losses} lệnh thua")
    
elif hasattr(strategy, 'trades') and len(strategy.trades) > 0:
    # Fallback: sử dụng trades từ Backtrader
    print(f"\nTổng số lệnh đã đóng: {len(strategy.trades)}")
    print("\n" + "-" * 90)
    print(f"{'STT':<5} {'Ngày Mua':<12} {'Giá Mua':<15} {'Ngày Bán':<12} {'Giá Bán':<15} {'Lợi nhuận':<15} {'Kết quả':<10}")
    print("-" * 90)
    
    for i, trade in enumerate(strategy.trades, 1):
        if trade.isclosed:
            entry_date = trade.dtopen.date() if hasattr(trade.dtopen, 'date') else str(trade.dtopen)
            entry_price = trade.price
            exit_date = trade.dtclose.date() if hasattr(trade.dtclose, 'date') else str(trade.dtclose)
            exit_price = trade.pnl / trade.size + trade.price if trade.size != 0 else 0
            pnlcomm = trade.pnlcomm
            
            result = "✅ THẮNG" if pnlcomm > 0 else "❌ THUA" if pnlcomm < 0 else "➖ HÒA"
            
            print(f"{i:<5} {str(entry_date):<12} {entry_price:>14,.0f} {str(exit_date):<12} {exit_price:>14,.0f} {pnlcomm:>14,.0f} {result:<10}")
    
    print("-" * 90)
else:
    print("\n⚠️ Không có thông tin chi tiết về các lệnh giao dịch.")
    print("   (Có thể do chiến lược không lưu lại thông tin trades)")


CHI TIẾT TỪNG LỆNH GIAO DỊCH

Tổng số lệnh đã đóng: 1

------------------------------------------------------------------------------------------
STT   Ngày Mua     Giá Mua         Ngày Bán     Giá Bán         Lợi nhuận       Kết quả   
------------------------------------------------------------------------------------------
1     739288.0             24,575 739341.0                  0         85,089 ✅ THẮNG   
------------------------------------------------------------------------------------------

Tóm tắt: 1 lệnh thắng, 0 lệnh thua


# 📚 GIẢI THÍCH CÁCH TÍNH LỆNH THẮNG/THUA (THEO LOGIC FOREX)

## 🔍 CÁCH TÍNH LỆNH THẮNG/THUA:

### 1. **LỆNH LONG (MUA - BUY):**
   - **MỞ LỆNH**: Mua ở giá thấp (Entry Price)
   - **ĐÓNG LỆNH**: Bán ở giá cao hơn (Exit Price)
   - **THẮNG** ✅: Khi Exit Price > Entry Price → Lợi nhuận > 0
   - **THUA** ❌: Khi Exit Price < Entry Price → Lỗ < 0

   **Công thức:**
   ```
   PnL (Profit/Loss) = (Exit Price - Entry Price) × Số lượng - Phí giao dịch
   ```

### 2. **LỆNH SHORT (BÁN - SELL):**
   - **MỞ LỆNH**: Bán ở giá cao (Entry Price) 
   - **ĐÓNG LỆNH**: Mua lại ở giá thấp hơn (Exit Price)
   - **THẮNG** ✅: Khi Exit Price < Entry Price → Lợi nhuận > 0
   - **THUA** ❌: Khi Exit Price > Entry Price → Lỗ < 0

   **Công thức:**
   ```
   PnL (Profit/Loss) = (Entry Price - Exit Price) × Số lượng - Phí giao dịch
   ```

### 3. **VÍ DỤ CỤ THỂ:**

#### Ví dụ 1: LỆNH LONG THẮNG
- Mua 100 cổ phiếu SSI ở giá: **24,429 VNĐ**
- Bán 100 cổ phiếu SSI ở giá: **25,697 VNĐ**
- Phí giao dịch: 0.15% mỗi lần = ~3,700 VNĐ (mua) + ~3,900 VNĐ (bán) = ~7,600 VNĐ
- **Lợi nhuận** = (25,697 - 24,429) × 100 - 7,600 = **126,800 - 7,600 = 119,200 VNĐ** ✅ THẮNG

#### Ví dụ 2: LỆNH LONG THUA
- Mua 100 cổ phiếu SSI ở giá: **25,000 VNĐ**
- Bán 100 cổ phiếu SSI ở giá: **24,000 VNĐ**
- Phí giao dịch: ~7,500 VNĐ
- **Lỗ** = (24,000 - 25,000) × 100 - 7,500 = **-100,000 - 7,500 = -107,500 VNĐ** ❌ THUA

#### Ví dụ 3: LỆNH SHORT THẮNG (Forex/Crypto)
- Bán 1 lot EUR/USD ở giá: **1.1000**
- Mua lại 1 lot EUR/USD ở giá: **1.0950**
- **Lợi nhuận** = (1.1000 - 1.0950) × 100,000 = **50 pips = $500** ✅ THẮNG

#### Ví dụ 4: LỆNH SHORT THUA (Forex/Crypto)
- Bán 1 lot EUR/USD ở giá: **1.1000**
- Mua lại 1 lot EUR/USD ở giá: **1.1050**
- **Lỗ** = (1.1000 - 1.1050) × 100,000 = **-50 pips = -$500** ❌ THUA

### 4. **CÁC CHỈ SỐ QUAN TRỌNG:**

- **Win Rate (%)** = (Số lệnh thắng / Tổng số lệnh) × 100
- **Profit Factor** = Tổng lợi nhuận từ lệnh thắng / Tổng lỗ từ lệnh thua (giá trị tuyệt đối)
- **Risk/Reward Ratio** = Lợi nhuận trung bình / Lỗ trung bình (giá trị tuyệt đối)
- **Expectancy** = (Win Rate × Avg Win) - (Loss Rate × Avg Loss)

### 5. **LƯU Ý:**
- Phí giao dịch (commission) được trừ vào lợi nhuận
- Lệnh được tính là THẮNG khi PnL > 0 sau khi trừ phí
- Lệnh được tính là THUA khi PnL < 0 sau khi trừ phí
- Lệnh HÒA khi PnL = 0 (rất hiếm)
